In [3]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install streamlit

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import numpy as np

movies = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')

print("Movies:", movies.shape)
print("Credits:", credits.shape)

Movies: (4803, 20)
Credits: (4803, 4)


In [7]:
# Merge both dataframes on the 'title' column
movies = movies.merge(credits, on='title')

# Check the new shape of our merged dataset
print("Merged Dataset Shape:", movies.shape)

# See the new column list
movies.head(1)

Merged Dataset Shape: (4809, 23)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [8]:
# Keep only the important columns for content-based filtering
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

# Look at the first 5 rows of our trimmed dataset
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [9]:
# 1. Check for missing values in each column
print("Missing values before cleaning:")
print(movies.isnull().sum())

# 2. Drop rows where 'overview' is missing (since we can't recommend without an overview)
movies.dropna(inplace=True)

# 3. Check for duplicate rows and remove them
print("\nDuplicate rows found:", movies.duplicated().sum())
movies.drop_duplicates(inplace=True)

# 4. Verify the final shape
print("\nCleaned Dataset Shape:", movies.shape)

Missing values before cleaning:
movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

Duplicate rows found: 0

Cleaned Dataset Shape: (4806, 7)


In [10]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [11]:
import ast

# Define a helper function to extract names from the JSON-like strings
def convert_features(obj):
    names_list = []
    # ast.literal_eval converts the string '[...]' into an actual Python list [...]
    for i in ast.literal_eval(obj):
        names_list.append(i['name'])
    return names_list

# Apply the function to the 'genres' column
movies['genres'] = movies['genres'].apply(convert_features)

# Apply the function to the 'keywords' column
movies['keywords'] = movies['keywords'].apply(convert_features)

# Let's see the result for the first movie
movies[['title', 'genres', 'keywords']].head()

,title,genres,keywords
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon..."
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ..."
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi..."
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i..."
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel..."


In [12]:
# Helper function to extract only the top 3 actors from the cast column
def convert_cast(obj):
    actors_list = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            actors_list.append(i['name'])
            counter += 1
        else:
            break
    return actors_list

# Helper function to extract ONLY the director from the crew column
def fetch_director(obj):
    director_list = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            director_list.append(i['name'])
            break
    return director_list

# Apply the cast function
movies['cast'] = movies['cast'].apply(convert_cast)

# Apply the crew/director function
movies['crew'] = movies['crew'].apply(fetch_director)

# Look at the first 3 rows to confirm everything is clean
movies[['title', 'cast', 'crew']].head(3)

,title,cast,crew
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,Spectre,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]


In [13]:
# Function to collapse spaces inside lists of strings
def collapse_spaces(lst):
    return [i.replace(" ", "") for i in lst]

# Apply space collapsing to our 4 target columns
movies['genres'] = movies['genres'].apply(collapse_spaces)
movies['keywords'] = movies['keywords'].apply(collapse_spaces)
movies['cast'] = movies['cast'].apply(collapse_spaces)
movies['crew'] = movies['crew'].apply(collapse_spaces)

# Also, convert the 'overview' column from a string into a list of words 
# so we can easily combine it with the other lists later.
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# Take a look at the updated first row
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]


In [15]:
# 1. Combine all 5 list columns into a single 'tags' column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# 2. Create a clean final dataframe with only the columns we need moving forward
new_df = movies[['movie_id', 'title', 'tags']]

# 3. Convert the list of tags back into a single string separated by spaces
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

# 4. Convert all text in tags to lowercase to keep everything standardized
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

# Check our final processed dataframe!
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [17]:
# Install NLTK directly inside your notebook if you don't have it
!pip install nltk

import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

# Create a helper function to stem every word in a text string
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

# Apply stemming to our tags column
new_df['tags'] = new_df['tags'].apply(stem)

print("Stemming complete! Text has been normalized.")

Defaulting to user installation because normal site-packages is not writeable
Stemming complete! Text has been normalized.


In [18]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer: keep top 5000 words, remove English stop words
cv = CountVectorizer(max_features=5000, stop_words='english')

# Transform our tags text into a numerical array
vectors = cv.fit_transform(new_df['tags']).toarray()

# Check the shape of our vector matrix
print("Vectors Matrix Shape:", vectors.shape)

Vectors Matrix Shape: (4806, 5000)


In [19]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute the similarity scores between all movie vectors
similarity = cosine_similarity(vectors)

# Check the matrix structure
print("Similarity Matrix Shape:", similarity.shape)

Similarity Matrix Shape: (4806, 4806)


In [20]:
def recommend(movie):
    # 1. Find the index of the requested movie
    movie_index = new_df[new_df['title'] == movie].index[0]
    
    # 2. Get the row of similarity scores for that movie
    distances = similarity[movie_index]
    
    # 3. Pair scores with indices, sort them in descending order, and pick the top 5 (excluding the movie itself)
    movies_list = sorted(list(enumerate(distances)), key=lambda x: x[1], reverse=True)[1:6]
    
    # 4. Print out the top 5 recommendations
    print(f"Recommendations for '{movie}':\n")
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

# --- TEST YOUR WORK! ---
recommend('Avatar')

Recommendations for 'Avatar':

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [21]:
import pickle

# 1. Save the processed movie dataframe as a dictionary (easier to load into Streamlit)
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))

# 2. Save the massive similarity matrix
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("Files saved successfully! 'movie_dict.pkl' and 'similarity.pkl' are now in your folder.")

Files saved successfully! 'movie_dict.pkl' and 'similarity.pkl' are now in your folder.
